# **UCI HAR - Fine-tuning**

In [1]:
import numpy as np
import pandas as pd
import tensorflow as tf
import os
from tensorflow.keras import backend as K
from tensorflow.keras import layers, Model
from tensorflow.keras.applications import VGG16
from tensorflow.keras.applications.vgg16 import preprocess_input

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, classification_report

In [2]:
SIGNAL_NAMES = [
    "body_acc_x_",
    "body_acc_y_",
    "body_acc_z_",
    "body_gyro_x_",
    "body_gyro_y_",
    "body_gyro_z_",
    "total_acc_x_",
    "total_acc_y_",
    "total_acc_z_"
]

LABELS = [
    "WALKING",
    "WALKING_UPSTAIRS",
    "WALKING_DOWNSTAIRS",
    "SITTING",
    "STANDING",
    "LAYING"
]

In [3]:
NUM_CLASSES = 6
BATCH_SIZE = 64

HEAD_EPOCHS = 30
FINETUNE_EPOCHS = 40

IMG_SIZE = 128

LR_HEAD = 1e-3
LR_FINETUNE = 1e-5

In [4]:
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

In [5]:
TRAIN = "train/"
TEST = "test/"

BASE_DIR = "/content/drive/MyDrive/AI_assignment/UCI_HAR_Dataset/"

In [6]:
def load_signals(split):
    signal_dir = os.path.join(BASE_DIR, split, "Inertial_Signals")
    signals = []

    for name in SIGNAL_NAMES:
        file_path = os.path.join(signal_dir, f"{name}{split}.txt")
        data = np.loadtxt(file_path)
        signals.append(data)

    # (N, 128, 9)
    signals = np.stack(signals, axis=-1).astype(np.float32)
    return signals


def load_labels(split):
    label_path = os.path.join(BASE_DIR, split, f"y_{split}.txt")
    labels = np.loadtxt(label_path).astype(np.int64)

    # UCI-HAR labels are 1~6, convert to 0~5
    labels = labels - 1
    return labels


X_train = load_signals("train")
y_train = load_labels("train")

X_test = load_signals("test")
y_test = load_labels("test")

print("Original train:", X_train.shape, y_train.shape)
print("Original test :", X_test.shape, y_test.shape)

Original train: (7352, 128, 9) (7352,)
Original test : (2947, 128, 9) (2947,)


In [7]:
N_train, T, C = X_train.shape
N_test = X_test.shape[0]

scaler = StandardScaler()

X_train_2d = X_train.reshape(-1, C)
X_test_2d = X_test.reshape(-1, C)

X_train = scaler.fit_transform(X_train_2d).reshape(N_train, T, C).astype(np.float32)
X_test = scaler.transform(X_test_2d).reshape(N_test, T, C).astype(np.float32)

print("After standardization:", X_train.shape, X_test.shape)

After standardization: (7352, 128, 9) (2947, 128, 9)


In [8]:
# Convert Time-series Window to VGG Input Image
# ============================================================
# Original UCI-HAR window: (128, 9)
# Converted VGG input    : (128, 128, 3)
#
# Step:
# 1) expand channel: (128, 9, 1)
# 2) resize to      : (128, 128, 1)
# 3) repeat channel : (128, 128, 3)
# 4) scale to image-like range and apply VGG preprocess_input
# ============================================================

def timeseries_to_vgg_image(x, y):
    x = tf.expand_dims(x, axis=-1)               # (128, 9, 1)
    x = tf.image.resize(x, (IMG_SIZE, IMG_SIZE)) # (128, 128, 1)
    x = tf.repeat(x, repeats=3, axis=-1)         # (128, 128, 3)

    # Convert standardized signal to pseudo image intensity range
    x = tf.clip_by_value(x, -3.0, 3.0)
    x = (x + 3.0) / 6.0
    x = x * 255.0

    x = preprocess_input(x)
    return x, y


def build_dataset(X, y, training=True):
    ds = tf.data.Dataset.from_tensor_slices((X, y))

    if training:
        ds = ds.shuffle(buffer_size=len(X), seed=SEED)

    ds = ds.map(timeseries_to_vgg_image, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(BATCH_SIZE)
    ds = ds.prefetch(tf.data.AUTOTUNE)

    return ds


train_ds = build_dataset(X_train, y_train, training=True)
test_ds = build_dataset(X_test, y_test, training=False)

In [9]:
# Build ImageNet Pretrained VGG16 Classifier
# ============================================================

def build_pretrained_vgg16_model(num_classes=6):
    inputs = layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3))

    base_model = VGG16(
        include_top=False,
        weights="imagenet",
        input_tensor=inputs
    )

    # First stage: freeze pretrained VGG16
    base_model.trainable = False

    x = base_model.output
    x = layers.GlobalAveragePooling2D()(x)

    x = layers.Dense(256, activation="relu")(x)
    x = layers.Dropout(0.5)(x)

    x = layers.Dense(128, activation="relu")(x)
    x = layers.Dropout(0.3)(x)

    outputs = layers.Dense(num_classes, activation="softmax")(x)

    model = Model(inputs=inputs, outputs=outputs, name="Pretrained_VGG16_UCIHAR")

    return model, base_model


model, base_model = build_pretrained_vgg16_model(num_classes=NUM_CLASSES)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=LR_HEAD),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

58889256/58889256 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


Model: "Pretrained_VGG16_UCIHAR"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 128, 128, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_conv1 (Conv2D)           │ (None, 128, 128, 64)   │         1,792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_conv2 (Conv2D)           │ (None, 128, 128, 64)   │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_pool (MaxPooling2D)      │ (None, 64, 64, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_conv1 (Conv2D)           │ (None, 64, 64, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_conv2 (Conv2D)           │ (None, 64, 64, 128)    │       147,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_pool (MaxPooling2D)      │ (None, 32, 32, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv1 (Conv2D)           │ (None, 32, 32, 256)    │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv2 (Conv2D)           │ (None, 32, 32, 256)    │       590,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv3 (Conv2D)           │ (None, 32, 32, 256)    │       590,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_pool (MaxPooling2D)      │ (None, 16, 16, 256)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv1 (Conv2D)           │ (None, 16, 16, 512)    │     1,180,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv2 (Conv2D)           │ (None, 16, 16, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv3 (Conv2D)           │ (None, 16, 16, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_pool (MaxPooling2D)      │ (None, 8, 8, 512)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv1 (Conv2D)           │ (None, 8, 8, 512)      │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv2 (Conv2D)           │ (None, 8, 8, 512)      │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv3 (Conv2D)           │ (None, 8, 8, 512)      │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_pool (MaxPooling2D)      │ (None, 4, 4, 512)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 512)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │       131,328 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼─────────────

 Total params: 14,879,686 (56.76 MB)

 Trainable params: 164,998 (644.52 KB)

 Non-trainable params: 14,714,688 (56.13 MB)

In [10]:
# Stage 1: Train Classifier Head Only
# ============================================================

print("\n================ Stage 1: Train Classifier Head ================\n")

callbacks_stage1 = [
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="loss",
        factor=0.5,
        patience=5,
        min_lr=1e-5,
        verbose=1
    )
]

model.fit(
    train_ds,
    epochs=HEAD_EPOCHS,
    callbacks=callbacks_stage1,
    verbose=1
)


================ Stage 1: Train Classifier Head ================

Epoch 1/30
115/115 ━━━━━━━━━━━━━━━━━━━━ 40s 212ms/step - accuracy: 0.6817 - loss: 0.9493 - learning_rate: 0.0010
Epoch 2/30
115/115 ━━━━━━━━━━━━━━━━━━━━ 14s 120ms/step - accuracy: 0.8473 - loss: 0.3970 - learning_rate: 0.0010
Epoch 3/30
115/115 ━━━━━━━━━━━━━━━━━━━━ 14s 123ms/step - accuracy: 0.8747 - loss: 0.3253 - learning_rate: 0.0010
Epoch 4/30
115/115 ━━━━━━━━━━━━━━━━━━━━ 15s 129ms/step - accuracy: 0.8966 - loss: 0.2713 - learning_rate: 0.0010
Epoch 5/30
115/115 ━━━━━━━━━━━━━━━━━━━━ 16s 138ms/step - accuracy: 0.9008 - loss: 0.2547 - learning_rate: 0.0010
Epoch 6/30
115/115 ━━━━━━━━━━━━━━━━━━━━ 16s 140ms/step - accuracy: 0.9123 - loss: 0.2263 - learning_rate: 0.0010
Epoch 7/30
115/115 ━━━━━━━━━━━━━━━━━━━━ 20s 131ms/step - accuracy: 0.9165 - loss: 0.2079 - learning_rate: 0.0010
Epoch 8/30
115/115 ━━━━━━━━━━━━━━━━━━━━ 20s 130ms/step - accuracy: 0.9221 - loss: 0.1922 - learning_rate: 0.0010
Epoch 9/30
115/115 ━━━━━━━━━━

In [11]:
# Stage 2: Fine-tune Last VGG Block
# ============================================================
# Fine-tuning only the last convolutional block avoids destroying
# the pretrained ImageNet features too aggressively.
# ============================================================

base_model.trainable = True

for layer in base_model.layers:
    if layer.name.startswith("block5"):
        layer.trainable = True
    else:
        layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=LR_FINETUNE),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

In [12]:
print("\n================ Stage 2: Fine-tune VGG Block5 ================\n")

callbacks_stage2 = [
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="loss",
        factor=0.5,
        patience=5,
        min_lr=1e-7,
        verbose=1
    )
]

model.fit(
    train_ds,
    epochs=FINETUNE_EPOCHS,
    callbacks=callbacks_stage2,
    verbose=1
)


================ Stage 2: Fine-tune VGG Block5 ================

Epoch 1/40
115/115 ━━━━━━━━━━━━━━━━━━━━ 31s 199ms/step - accuracy: 0.9646 - loss: 0.0921 - learning_rate: 1.0000e-05
Epoch 2/40
115/115 ━━━━━━━━━━━━━━━━━━━━ 19s 165ms/step - accuracy: 0.9804 - loss: 0.0573 - learning_rate: 1.0000e-05
Epoch 3/40
115/115 ━━━━━━━━━━━━━━━━━━━━ 18s 159ms/step - accuracy: 0.9820 - loss: 0.0491 - learning_rate: 1.0000e-05
Epoch 4/40
115/115 ━━━━━━━━━━━━━━━━━━━━ 18s 154ms/step - accuracy: 0.9863 - loss: 0.0374 - learning_rate: 1.0000e-05
Epoch 5/40
115/115 ━━━━━━━━━━━━━━━━━━━━ 18s 153ms/step - accuracy: 0.9886 - loss: 0.0312 - learning_rate: 1.0000e-05
Epoch 6/40
115/115 ━━━━━━━━━━━━━━━━━━━━ 18s 156ms/step - accuracy: 0.9833 - loss: 0.0419 - learning_rate: 1.0000e-05
Epoch 7/40
115/115 ━━━━━━━━━━━━━━━━━━━━ 18s 158ms/step - accuracy: 0.9857 - loss: 0.0387 - learning_rate: 1.0000e-05
Epoch 8/40
115/115 ━━━━━━━━━━━━━━━━━━━━ 20s 157ms/step - accuracy: 0.9928 - loss: 0.0211 - learning_rate: 1.0000e-0

In [13]:
# Test Evaluation
# ============================================================

print("\n================ Final Test Evaluation ================\n")

y_prob = model.predict(test_ds)
y_pred = np.argmax(y_prob, axis=1)

acc = accuracy_score(y_test, y_pred)
macro_f1 = f1_score(y_test, y_pred, average="macro")

print(f"Accuracy : {acc:.4f}")
print(f"Macro-F1 : {macro_f1:.4f}")

print("\nClassification Report")
print(classification_report(
    y_test,
    y_pred,
    target_names=[
        "WALKING",
        "WALKING_UPSTAIRS",
        "WALKING_DOWNSTAIRS",
        "SITTING",
        "STANDING",
        "LAYING"
    ],
    digits=4
))


================ Final Test Evaluation ================

47/47 ━━━━━━━━━━━━━━━━━━━━ 10s 185ms/step
Accuracy : 0.9410
Macro-F1 : 0.9412

Classification Report
                    precision    recall  f1-score   support

           WALKING     0.9836    0.9698    0.9766       496
  WALKING_UPSTAIRS     0.9707    0.9851    0.9779       471
WALKING_DOWNSTAIRS     0.9619    0.9619    0.9619       420
           SITTING     0.8867    0.8289    0.8568       491
          STANDING     0.8791    0.9023    0.8905       532
            LAYING     0.9676    1.0000    0.9835       537

          accuracy                         0.9410      2947
         macro avg     0.9416    0.9413    0.9412      2947
      weighted avg     0.9405    0.9410    0.9405      2947

